In [16]:
from collections import Counter, defaultdict
import math

In [17]:
corpus = [
    "I love Deep Learning",
    "Deep Learing requires high computational power",
    "Deep Learning is used for image and video processing",
    "Deep Learning is used for speech processing",
    "Deep Learning is used for text processing",
]

print("Corpus: ", corpus)


Corpus:  ['I love Deep Learning', 'Deep Learing requires high computational power', 'Deep Learning is used for image and video processing', 'Deep Learning is used for speech processing', 'Deep Learning is used for text processing']


In [18]:
def tokenize(sentence):
    return ["<s>"] + sentence.lower().split() + ["</s>"]



corpus_tokenized = [tokenize(sentence) for sentence in corpus]

all_words = [w for sent in corpus_tokenized for w in sent]
vocab = set(all_words) - {"<s>", "</s>"}
V = len(vocab)

print(f"Corpus: {len(corpus)}")
print(f"Vocab: {len(vocab)}")
print(f"Vocab: {vocab}")
print(f"Vocab: {V}")
print(f"Tokens : {' | '.join(sorted(vocab))}")

Corpus: 5
Vocab: 18
Vocab: {'processing', 'speech', 'used', 'learning', 'is', 'video', 'requires', 'text', 'and', 'high', 'i', 'power', 'deep', 'image', 'learing', 'computational', 'for', 'love'}
Vocab: 18
Tokens : and | computational | deep | for | high | i | image | is | learing | learning | love | power | processing | requires | speech | text | used | video


In [26]:
unigrams = Counter(all_words)
bigram = defaultdict(Counter)

for sent in corpus_tokenized:
    for w1, w2 in zip(sent[:-1], sent[1:]):
        bigram[w1][w2] += 1

total_tokens = sum(unigrams.values())

for word, count in unigrams.most_common(10):
    print(f"P({word:20s}) = {count}/{total_tokens} = {count/total_tokens:.4f}")

P(<s>                 ) = 5/43 = 0.1163
P(deep                ) = 5/43 = 0.1163
P(</s>                ) = 5/43 = 0.1163
P(learning            ) = 4/43 = 0.0930
P(is                  ) = 3/43 = 0.0698
P(used                ) = 3/43 = 0.0698
P(for                 ) = 3/43 = 0.0698
P(processing          ) = 3/43 = 0.0698
P(i                   ) = 1/43 = 0.0233
P(love                ) = 1/43 = 0.0233


In [27]:
def p_bigram_mle(w1,w2):
    """maximum Likelihood Estimate(no smoothing)"""
    if unigrams[w1] == 0:
        return 0.0
    return bigram[w1][w2] / unigrams[w1]


def p_bigram_laplace(w1, w2):
    """Laplace (Add-1) Smoothing."""
    return (bigram[w1][w2] + 1) / (unigrams[w1] + V)


def p_bigram_additive(w1, w2, k=0.5):
    """Additive (Add-k) Smoothing."""
    return (bigram[w1][w2] + k) / (unigrams[w1] + k * V)    

In [29]:
test_bigrams = [
    ("i", "love"),
    ("deep", "learning"),
    ("learning", "is"),
    ("i","hate"),
    ("amit","het")
]

print(f"{'Bigram':<28} {'MLE':>10} {'Laplace':>10} {'Add-k(0.5)':>12}")

for w1, w2 in test_bigrams:
    mle = p_bigram_mle(w1, w2)
    laplace = p_bigram_laplace(w1, w2)
    additive = p_bigram_additive(w1, w2, k=0.5)
    tag  = "  ← ZERO PROB" if mle == 0 else ""
    print(f"  P({w2}|{w1}){'':<{18 - len(w1) - len(w2)}} {mle:>10.4f} {laplace:>10.4f} {additive:>12.4f}{tag}")



Bigram                              MLE    Laplace   Add-k(0.5)
  P(love|i)                  1.0000     0.1053       0.1500
  P(learning|deep)           0.8000     0.2174       0.3214
  P(is|learning)             0.7500     0.1818       0.2692
  P(hate|i)                  0.0000     0.0526       0.0500  ← ZERO PROB
  P(het|amit)                0.0000     0.0556       0.0556  ← ZERO PROB


In [33]:
def sentence_probability(sentence, smoothing="mle", k=0.5):
    tokens = tokenize(sentence)
    prob = 1.0
    for w1, w2 in zip(tokens[:-1], tokens[1:]):
        if smoothing == "mle":
            prob *= p_bigram_mle(w1, w2)
        elif smoothing == "laplace":
            prob *= p_bigram_laplace(w1, w2)
        elif smoothing == "additive":
            prob *= p_bigram_additive(w1, w2, k)
    return prob

test_sentences = [
    "I love Deep Learning",
    "Deep Learing requires high computational power",
    "Deep Learning is used for image and video processing",
    "Deep Learning is used for speech processing",
    "Deep Learning is used for text processing",
    "I hate Deep Learning",
    "Amit Het Deep Learning"
]

print(f"{'Sentence':<60} {'MLE':>10} {'Laplace':>10} {'Add-k(0.5)':>12}")

for sent in test_sentences:
    mle_prob = sentence_probability(sent, smoothing="mle")
    laplace_prob = sentence_probability(sent, smoothing="laplace")
    additive_prob = sentence_probability(sent, smoothing="additive", k=0.5)
    tag = "  ← ZERO PROB" if mle_prob == 0 else ""
    print(f"{sent:<60} {mle_prob:>10.4e} {laplace_prob:>10.4e} {additive_prob:>12.4e}{tag}")


Sentence                                                            MLE    Laplace   Add-k(0.5)
I love Deep Learning                                         4.0000e-02 1.9042e-05   8.9408e-05
Deep Learing requires high computational power               1.6000e-01 2.4430e-07   2.6152e-06
Deep Learning is used for image and video processing         1.6000e-01 6.5960e-09   2.9116e-07
Deep Learning is used for speech processing                  1.6000e-01 5.9529e-07   1.2941e-05
Deep Learning is used for text processing                    1.6000e-01 5.9529e-07   1.2941e-05
I hate Deep Learning                                         0.0000e+00 5.0249e-06   1.1038e-05  ← ZERO PROB
Amit Het Deep Learning                                       0.0000e+00 2.6520e-06   4.0882e-06  ← ZERO PROB


In [39]:
k_values = [0.1, 0.5, 1.0, 2.0]
print(f"{'k':>8} {'Sentence Probability':>20}")
for k in k_values:
        prob = sentence_probability("I hate Deep Learning", smoothing="additive", k=k)
        print(f"{k:>8} {prob:>20.4e}")


       k Sentence Probability
     0.1           3.6702e-05
     0.5           1.1038e-05
     1.0           5.0249e-06
     2.0           2.4117e-06
